In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import pandas as pd
import scipy.stats as stats
import sklearn.metrics
import itertools

In [2]:
def load_performance_metrics_for_T_tests(approach, dataset, target_metric, trial):
  
  performance = list()

  random_states = [x for x in os.listdir(f'../../outputs/{approach.replace("-", "_").lower()}/{dataset}-{approach}/{trial}') if os.path.isdir(f'../../outputs/{approach.replace("-", "_").lower()}/{dataset}-{approach}/{trial}/{x}')]
  if len(random_states) < 10:
    print('Not enough random states...')

  for random_state in random_states:
    df = pd.read_csv(f'../../outputs/{approach.replace("-", "_").lower()}/{dataset}-{approach}/{trial}/{random_state}/predictions.csv')
    for split in ['validation', 'test']:
      df_split = df[df['split'] == split]
      accuracy = sklearn.metrics.accuracy_score(df_split['real'], df_split['prediction'])
      f1_score = sklearn.metrics.f1_score(df_split['real'], df_split['prediction'], average = 'macro')
      precision = sklearn.metrics.precision_score(df_split['real'], df_split['prediction'], average = 'macro')
      recall = sklearn.metrics.recall_score(df_split['real'], df_split['prediction'], average = 'macro')
      performance.append((trial, random_state, split, accuracy, f1_score, precision, recall))  
  return pd.DataFrame(performance, columns = ['trial', 'random_state', 'split', 'accuracy', 'f1_score', 'precision', 'recall'])[['random_state', 'split', target_metric]].rename(columns = {target_metric : 'performance'})

In [3]:
def get_all_performance_metrics_for_T_tests(approach, datasets, target_metrics, trials):
  l = list()
  for dataset, target_metric, trial in zip(datasets, target_metrics, trials):
    l.append(
      load_performance_metrics_for_T_tests(
        approach = approach,
        dataset = dataset,
        trial = trial,
        target_metric = target_metric,
      ).assign(
        approach = approach,
        dataset = dataset,
      )
    )
  return pd.concat(l, axis = 0).reset_index(drop = True)

In [4]:
standard_vs_truncated = pd.concat([
  get_all_performance_metrics_for_T_tests(
    approach = 'Fine-tuning-Truncated',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [0, 78, 71, 76]
  ),
  get_all_performance_metrics_for_T_tests(
    approach = 'Fine-tuning',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [104, 0, 0, 52]
  )
])

In [5]:
for dataset in ['SST-2', 'Ohsumed', 'R8', 'IMDb']:
  print('-' * 10, dataset, '-' * 10)
  for i, split in enumerate(['validation', 'test']):
    print('>>', split)
    split_df = standard_vs_truncated[(standard_vs_truncated['dataset'] == dataset) & (standard_vs_truncated['split'] == split)].drop(columns = ['split']) \
      .pivot(index = ['dataset', 'random_state'], columns = 'approach', values = 'performance') \
      .reset_index() \
      .sort_values(by = ['dataset', 'random_state'])
    
    for (x, y) in itertools.combinations(['Fine-tuning', 'Fine-tuning-Truncated'], 2):
      # H0: Mean Grouped and Surrogate scores are equal
      # H1: Mean Grouped and Surrogate scores are not equal
      statistic, p_value = stats.ttest_ind(split_df[x], split_df[y])
      # 95% confidence -- p-value < 0.05 => reject the null hypothesis (H0), i.e., the true mean test score is different between the approaches
      print(f'{x} - {y}:', 'Statistic:', statistic, 'P-value:', p_value, 'Reject H0?', p_value < 0.05)
  print('')

---------- SST-2 ----------
>> validation
Fine-tuning - Fine-tuning-Truncated: Statistic: -0.5318965725889424 P-value: 0.6013015272697095 Reject H0? False
>> test
Fine-tuning - Fine-tuning-Truncated: Statistic: -1.0912431619905385 P-value: 0.2895581238492713 Reject H0? False

---------- Ohsumed ----------
>> validation
Fine-tuning - Fine-tuning-Truncated: Statistic: -1.6940802382741817 P-value: 0.10748234652602119 Reject H0? False
>> test
Fine-tuning - Fine-tuning-Truncated: Statistic: -3.012320328613854 P-value: 0.007482163150123928 Reject H0? True

---------- R8 ----------
>> validation
Fine-tuning - Fine-tuning-Truncated: Statistic: -4.766210660170765 P-value: 0.00015438077854903448 Reject H0? True
>> test
Fine-tuning - Fine-tuning-Truncated: Statistic: -7.301497567386523 P-value: 8.777424387226543e-07 Reject H0? True

---------- IMDb ----------
>> validation
Fine-tuning - Fine-tuning-Truncated: Statistic: 0.9230071547042693 P-value: 0.36821134901415387 Reject H0? False
>> test
Fine